In [2]:
import pandas as pd

df = pd.read_csv("../data/home-credit-default-risk/application_train.csv")

In [3]:
print(df.shape)

(307511, 122)


In [4]:
df.head()

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
4,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0


TARGET represents whether the applicant defaulted on the loan (1) or not (0). This will be the dependent variable for modeling Probability of Default (PD).

In [5]:
df["TARGET"].value_counts(normalize=True)

TARGET
0    0.919271
1    0.080729
Name: proportion, dtype: float64

The distribution of the target variable shows that approximately 8% of applicants defaulted, while 92% did not default.

This indicates that the dataset is highly imbalanced, which is common in credit risk modeling because most borrowers repay their loans. In later phases, a stratified train/test split will be used to maintain the same class distribution across datasets.

In [6]:
missing = df.isnull().mean().sort_values(ascending=False)
missing.head(20)

COMMONAREA_AVG              0.698723
COMMONAREA_MODE             0.698723
COMMONAREA_MEDI             0.698723
NONLIVINGAPARTMENTS_MEDI    0.694330
NONLIVINGAPARTMENTS_MODE    0.694330
NONLIVINGAPARTMENTS_AVG     0.694330
FONDKAPREMONT_MODE          0.683862
LIVINGAPARTMENTS_AVG        0.683550
LIVINGAPARTMENTS_MEDI       0.683550
LIVINGAPARTMENTS_MODE       0.683550
FLOORSMIN_MODE              0.678486
FLOORSMIN_AVG               0.678486
FLOORSMIN_MEDI              0.678486
YEARS_BUILD_AVG             0.664978
YEARS_BUILD_MODE            0.664978
YEARS_BUILD_MEDI            0.664978
OWN_CAR_AGE                 0.659908
LANDAREA_MEDI               0.593767
LANDAREA_AVG                0.593767
LANDAREA_MODE               0.593767
dtype: float64

## Approved Features

In [7]:
core_features = [
    # Demographics
    "CODE_GENDER", "CNT_CHILDREN", "NAME_FAMILY_STATUS",
    "CNT_FAM_MEMBERS", "NAME_EDUCATION_TYPE", "DAYS_BIRTH",

    # Employment & Income
    "NAME_INCOME_TYPE", "OCCUPATION_TYPE",
    "ORGANIZATION_TYPE", "DAYS_EMPLOYED",
    "AMT_INCOME_TOTAL",

    # Housing
    "FLAG_OWN_CAR", "OWN_CAR_AGE",
    "FLAG_OWN_REALTY", "NAME_HOUSING_TYPE",
    "DAYS_REGISTRATION", "DAYS_ID_PUBLISH",

    # Loan
    "NAME_CONTRACT_TYPE", "AMT_CREDIT",
    "AMT_ANNUITY", "AMT_GOODS_PRICE",

    # Bureau Signals
    "EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3",
    "AMT_REQ_CREDIT_BUREAU_HOUR",
    "AMT_REQ_CREDIT_BUREAU_DAY",
    "AMT_REQ_CREDIT_BUREAU_WEEK",
    "AMT_REQ_CREDIT_BUREAU_MON",
    "AMT_REQ_CREDIT_BUREAU_QRT",
    "AMT_REQ_CREDIT_BUREAU_YEAR",

    # Target
    "TARGET"
]

df = df[core_features]

In [8]:
df.shape
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 307511 entries, 0 to 307510
Data columns (total 31 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   CODE_GENDER                 307511 non-null  object 
 1   CNT_CHILDREN                307511 non-null  int64  
 2   NAME_FAMILY_STATUS          307511 non-null  object 
 3   CNT_FAM_MEMBERS             307509 non-null  float64
 4   NAME_EDUCATION_TYPE         307511 non-null  object 
 5   DAYS_BIRTH                  307511 non-null  int64  
 6   NAME_INCOME_TYPE            307511 non-null  object 
 7   OCCUPATION_TYPE             211120 non-null  object 
 8   ORGANIZATION_TYPE           307511 non-null  object 
 9   DAYS_EMPLOYED               307511 non-null  int64  
 10  AMT_INCOME_TOTAL            307511 non-null  float64
 11  FLAG_OWN_CAR                307511 non-null  object 
 12  OWN_CAR_AGE                 104582 non-null  float64
 13  FLAG_OWN_REALT

Reduced from 122 columns to approximately 30 decision-time features to avoid leakage and improve interpretability.

## Engineer Business Ratios

In [9]:
df["AGE"] = -df["DAYS_BIRTH"] / 365
df["EMPLOYMENT_YEARS"] = df["DAYS_EMPLOYED"].replace(365243, 0) / -365

df["DTI_PROXY"] = df["AMT_ANNUITY"] / df["AMT_INCOME_TOTAL"]
df["LOAN_TO_INCOME"] = df["AMT_CREDIT"] / df["AMT_INCOME_TOTAL"]

Created engineered features to better capture repayment burden and income stress.

## Data Dictionary

In [10]:
import pandas as pd

data_dict = pd.DataFrame([
    {"Column": "TARGET", "Description": "Default flag (1 = default, 0 = no default)", "Type": "Binary", "Used For": "Label"},
    {"Column": "CODE_GENDER", "Description": "Applicant gender", "Type": "Categorical", "Used For": "Demographics"},
    {"Column": "CNT_CHILDREN", "Description": "Number of children", "Type": "Numeric", "Used For": "Demographics"},
    {"Column": "NAME_FAMILY_STATUS", "Description": "Applicant family status", "Type": "Categorical", "Used For": "Demographics"},
    {"Column": "CNT_FAM_MEMBERS", "Description": "Number of family members", "Type": "Numeric", "Used For": "Demographics"},
    {"Column": "NAME_EDUCATION_TYPE", "Description": "Highest education level", "Type": "Categorical", "Used For": "Demographics"},
    {"Column": "DAYS_BIRTH", "Description": "Applicant age in days before application", "Type": "Numeric", "Used For": "Age feature"},
    {"Column": "AGE", "Description": "Applicant age in years", "Type": "Numeric", "Used For": "Risk factor"},
    {"Column": "NAME_INCOME_TYPE", "Description": "Type of income source", "Type": "Categorical", "Used For": "Employment & Income"},
    {"Column": "OCCUPATION_TYPE", "Description": "Applicant occupation", "Type": "Categorical", "Used For": "Employment & Income"},
    {"Column": "ORGANIZATION_TYPE", "Description": "Type of employer organization", "Type": "Categorical", "Used For": "Employment & Income"},
    {"Column": "DAYS_EMPLOYED", "Description": "Days employed before application", "Type": "Numeric", "Used For": "Employment history"},
    {"Column": "AMT_INCOME_TOTAL", "Description": "Total annual income", "Type": "Numeric", "Used For": "Repayment capacity"},
    {"Column": "FLAG_OWN_CAR", "Description": "Whether applicant owns a car", "Type": "Categorical", "Used For": "Stability"},
    {"Column": "OWN_CAR_AGE", "Description": "Age of applicant's car", "Type": "Numeric", "Used For": "Stability"},
    {"Column": "FLAG_OWN_REALTY", "Description": "Whether applicant owns real estate", "Type": "Categorical", "Used For": "Housing stability"},
    {"Column": "NAME_HOUSING_TYPE", "Description": "Type of housing situation", "Type": "Categorical", "Used For": "Housing stability"},
    {"Column": "DAYS_REGISTRATION", "Description": "Days since registration at current address", "Type": "Numeric", "Used For": "Stability"},
    {"Column": "DAYS_ID_PUBLISH", "Description": "Days since ID document was last updated", "Type": "Numeric", "Used For": "Stability"},
    {"Column": "NAME_CONTRACT_TYPE", "Description": "Type of loan contract", "Type": "Categorical", "Used For": "Loan characteristic"},
    {"Column": "AMT_CREDIT", "Description": "Loan amount requested", "Type": "Numeric", "Used For": "Loan characteristic"},
    {"Column": "AMT_ANNUITY", "Description": "Loan annuity amount", "Type": "Numeric", "Used For": "Loan repayment burden"},
    {"Column": "AMT_GOODS_PRICE", "Description": "Price of goods for which loan is given", "Type": "Numeric", "Used For": "Loan characteristic"},
    {"Column": "DTI_PROXY", "Description": "Annuity divided by total income", "Type": "Numeric", "Used For": "Debt burden"},
    {"Column": "LOAN_TO_INCOME", "Description": "Credit amount divided by total income", "Type": "Numeric", "Used For": "Affordability"},
    {"Column": "EXT_SOURCE_1", "Description": "External credit risk score 1", "Type": "Numeric", "Used For": "Credit risk signal"},
    {"Column": "EXT_SOURCE_2", "Description": "External credit risk score 2", "Type": "Numeric", "Used For": "Credit risk signal"},
    {"Column": "EXT_SOURCE_3", "Description": "External credit risk score 3", "Type": "Numeric", "Used For": "Credit risk signal"},
    {"Column": "AMT_REQ_CREDIT_BUREAU_HOUR", "Description": "Credit bureau inquiries in last hour", "Type": "Numeric", "Used For": "Inquiry behavior"},
    {"Column": "AMT_REQ_CREDIT_BUREAU_DAY", "Description": "Credit bureau inquiries in last day", "Type": "Numeric", "Used For": "Inquiry behavior"},
    {"Column": "AMT_REQ_CREDIT_BUREAU_WEEK", "Description": "Credit bureau inquiries in last week", "Type": "Numeric", "Used For": "Inquiry behavior"},
    {"Column": "AMT_REQ_CREDIT_BUREAU_MON", "Description": "Credit bureau inquiries in last month", "Type": "Numeric", "Used For": "Inquiry behavior"},
    {"Column": "AMT_REQ_CREDIT_BUREAU_QRT", "Description": "Credit bureau inquiries in last quarter", "Type": "Numeric", "Used For": "Inquiry behavior"},
    {"Column": "AMT_REQ_CREDIT_BUREAU_YEAR", "Description": "Credit bureau inquiries in last year", "Type": "Numeric", "Used For": "Inquiry behavior"},
])

data_dict

,Column,Description,Type,Used For
0,TARGET,"Default flag (1 = default, 0 = no default)",Binary,Label
1,CODE_GENDER,Applicant gender,Categorical,Demographics
2,CNT_CHILDREN,Number of children,Numeric,Demographics
3,NAME_FAMILY_STATUS,Applicant family status,Categorical,Demographics
4,CNT_FAM_MEMBERS,Number of family members,Numeric,Demographics
5,NAME_EDUCATION_TYPE,Highest education level,Categorical,Demographics
6,DAYS_BIRTH,Applicant age in days before application,Numeric,Age feature
7,AGE,Applicant age in years,Numeric,Risk factor
8,NAME_INCOME_TYPE,Type of income source,Categorical,Employment & Income
9,OCCUPATION_TYPE,Applicant occupation,Categorical,Employment & Income


In [11]:
data_dict.to_csv("../data/data_dictionary.csv", index=False)